In [2]:
# -*- coding: utf-8 -*-
"""
RECONSTRUIR BASE CON GASTO - CORREGIDO
"""

import pandas as pd
import os
import numpy as np

# ============================================================
# CONFIGURACIÓN
# ============================================================
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# ============================================================
# 1. CARGAR MASTER 2024 y 2025
# ============================================================

print("="*70)
print("CREANDO NUEVA BASE CON GASTO (CORREGIDO)")
print("="*70)

df24 = pd.read_csv(os.path.join(base_resultados, "MASTER_2024.csv"), encoding="utf-8-sig", low_memory=False)
df25 = pd.read_csv(os.path.join(base_resultados, "MASTER_2025.csv"), encoding="utf-8-sig", low_memory=False)

print(f"✅ MASTER_2024: {len(df24):,} filas")
print(f"✅ MASTER_2025: {len(df25):,} filas")

# ============================================================
# 2. FUNCIÓN PARA CREAR LLAVE
# ============================================================

def construir_llave(df):
    df = df.copy()
    for c in ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]:
        df[f"_{c}"] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
    df["llave_persona"] = df[["_CONGLOME", "_VIVIENDA", "_HOGAR", "_CODPERSO"]].agg("-".join, axis=1)
    return df.drop(columns=["_CONGLOME", "_VIVIENDA", "_HOGAR", "_CODPERSO"])

df24 = construir_llave(df24)
df25 = construir_llave(df25)

# Emparejar
match = df24[["llave_persona"]].drop_duplicates().merge(
    df25[["llave_persona"]].drop_duplicates(), on="llave_persona", how="inner"
)
print(f"✅ Personas emparejadas: {len(match):,}")

# ============================================================
# 3. SELECCIONAR COLUMNAS 2024
# ============================================================

cols_2024 = ["llave_persona", "P203", "P207", "Edad", "Ocupado", "Informal",
             "TenenciaBilletera", "UsoBilletera", "CreditoFormal", "FACTOR07", 
             "P301A", "ESTRATO", "DOMINIO", "MIEPERHO", "CONGLOME",
             "GASHOG2"]

df_2024_sub = df24[df24["llave_persona"].isin(match["llave_persona"])][cols_2024].rename(
    columns={"CreditoFormal": "CreditoFormal_2024"}
)

# ============================================================
# 4. SELECCIONAR COLUMNAS 2025
# ============================================================

cols_2025 = ["llave_persona", "CreditoFormal"]
if "CreditoInformal" in df25.columns:
    cols_2025.append("CreditoInformal")

df_2025_sub = df25[cols_2025].rename(
    columns={"CreditoFormal": "CreditoFormal_2025", 
             "CreditoInformal": "CreditoInformal_2025"}
)

# ============================================================
# 5. UNIR Y FILTRAR
# ============================================================

panel_df = df_2024_sub.merge(df_2025_sub, on="llave_persona", how="left")
print(f"✅ Panel construido: {len(panel_df):,} filas")

# Filtros
panel_df["P203_num"] = pd.to_numeric(panel_df["P203"], errors="coerce")
panel_df["Ocupado_num"] = pd.to_numeric(panel_df["Ocupado"], errors="coerce")
panel_df["CreditoFormal_2024_num"] = pd.to_numeric(panel_df["CreditoFormal_2024"], errors="coerce")
panel_df["CreditoFormal_2025_num"] = pd.to_numeric(panel_df["CreditoFormal_2025"], errors="coerce")
panel_df["Informal_num"] = pd.to_numeric(panel_df["Informal"], errors="coerce")
panel_df["P301A_num"] = pd.to_numeric(panel_df["P301A"], errors="coerce")

# Excluir códigos 99
panel_df = panel_df[panel_df["P301A_num"] != 99]

# Filtros
panel_df = panel_df[panel_df["P203_num"] == 1]
panel_df = panel_df[panel_df["Ocupado_num"] == 1]
panel_df = panel_df[panel_df["CreditoFormal_2024_num"] == 0]

# Drop missing
vars_clave = ["Informal_num", "TenenciaBilletera", "UsoBilletera", "P207", "Edad", "P301A", "GASHOG2"]
panel_df = panel_df.dropna(subset=vars_clave)

# Crear variable dependiente
panel_df["NuevoCredito"] = (panel_df["CreditoFormal_2025_num"] == 1).astype(int)

print(f"✅ Muestra final: {len(panel_df):,} filas")

# ============================================================
# 6. RENOMBRAR
# ============================================================

diccionario_renombres = {
    "llave_persona": "id_persona",
    "P203": "jefe_hogar",
    "P207": "sexo",
    "Edad": "edad",
    "P301A": "nivel_educativo",
    "Ocupado": "ocupado",
    "Informal": "trabajador_informal",
    "TenenciaBilletera": "tiene_billetera",
    "UsoBilletera": "usa_billetera",
    "CreditoFormal_2024": "credito_formal_2024",
    "CreditoFormal_2025": "credito_formal_2025",
    "CreditoInformal_2025": "credito_informal_2025",
    "NuevoCredito": "nuevo_credito_formal",
    "FACTOR07": "factor_expansion",
    "ESTRATO": "estrato",
    "DOMINIO": "dominio",
    "MIEPERHO": "miembros_hogar",
    "CONGLOME": "conglomerado",
    "GASHOG2": "gasto_total_hogar",
}

columnas_a_renombrar = {k: v for k, v in diccionario_renombres.items() if k in panel_df.columns}
panel_df = panel_df.rename(columns=columnas_a_renombrar)

# Eliminar auxiliares
aux_cols = ["P203_num", "Ocupado_num", "CreditoFormal_2024_num", 
            "CreditoFormal_2025_num", "Informal_num", "P301A_num"]
panel_df = panel_df.drop(columns=[c for c in aux_cols if c in panel_df.columns])

# ============================================================
# 7. CORREGIR: CONVERTIR GASTO A NUMÉRICO
# ============================================================

# ¡ESTA ES LA LÍNEA CLAVE!
panel_df['gasto_total_hogar'] = pd.to_numeric(panel_df['gasto_total_hogar'], errors='coerce')

# ============================================================
# 8. CREAR log_gasto_percapita
# ============================================================

panel_df['gasto_percapita'] = panel_df['gasto_total_hogar'] / panel_df['miembros_hogar']
panel_df['log_gasto_percapita'] = np.log(panel_df['gasto_percapita'].clip(lower=1))

print(f"\n📊 Estadísticas de gasto:")
print(f"  • gasto_total_hogar media: {panel_df['gasto_total_hogar'].mean():.2f}")
print(f"  • gasto_percapita media: {panel_df['gasto_percapita'].mean():.2f}")
print(f"  • log_gasto_percapita media: {panel_df['log_gasto_percapita'].mean():.2f}")

# ============================================================
# 9. GUARDAR
# ============================================================

ruta_salida = os.path.join(base_resultados, "BASE_REGRESIONES_CON_GASTO.csv")
panel_df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
print(f"\n✅ GUARDADO: BASE_REGRESIONES_CON_GASTO.csv ({len(panel_df):,} filas, {len(panel_df.columns)} variables)")

print("\n" + "="*70)
print("VARIABLES FINALES")
print("="*70)
for i, col in enumerate(panel_df.columns, 1):
    print(f"{i:>3}. {col}")

print("\n" + "="*70)
print("VERIFICACIÓN RÁPIDA")
print("="*70)
print(f"🔍 usa_billetera: {panel_df['usa_billetera'].sum():,} ({panel_df['usa_billetera'].mean():.2%})")
print(f"🔍 nuevo_credito_formal: {panel_df['nuevo_credito_formal'].sum():,} ({panel_df['nuevo_credito_formal'].mean():.2%})")
print(f"🔍 log_gasto_percapita: media={panel_df['log_gasto_percapita'].mean():.2f}")

CREANDO NUEVA BASE CON GASTO (CORREGIDO)
✅ MASTER_2024: 117,721 filas
✅ MASTER_2025: 115,145 filas
✅ Personas emparejadas: 32,079
✅ Panel construido: 32,079 filas
✅ Muestra final: 6,358 filas

📊 Estadísticas de gasto:
  • gasto_total_hogar media: 33723.83
  • gasto_percapita media: 10911.15
  • log_gasto_percapita media: 9.15

✅ GUARDADO: BASE_REGRESIONES_CON_GASTO.csv (6,358 filas, 21 variables)

VARIABLES FINALES
  1. id_persona
  2. jefe_hogar
  3. sexo
  4. edad
  5. ocupado
  6. trabajador_informal
  7. tiene_billetera
  8. usa_billetera
  9. credito_formal_2024
 10. factor_expansion
 11. nivel_educativo
 12. estrato
 13. dominio
 14. miembros_hogar
 15. conglomerado
 16. gasto_total_hogar
 17. credito_formal_2025
 18. credito_informal_2025
 19. nuevo_credito_formal
 20. gasto_percapita
 21. log_gasto_percapita

VERIFICACIÓN RÁPIDA
🔍 usa_billetera: 872 (13.72%)
🔍 nuevo_credito_formal: 588 (9.25%)
🔍 log_gasto_percapita: media=9.15


In [3]:
# -*- coding: utf-8 -*-
"""
VERIFICAR MISSING EN BASE_REGRESIONES_CON_GASTO.csv
"""

import pandas as pd
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# ============================================================
# CARGAR NUEVA BASE
# ============================================================

ruta = os.path.join(base_resultados, "BASE_REGRESIONES_CON_GASTO.csv")
df = pd.read_csv(ruta, encoding="utf-8-sig")

print("="*70)
print("ANÁLISIS DE MISSING - BASE_REGRESIONES_CON_GASTO.csv")
print("="*70)
print(f"✅ Base cargada: {len(df):,} filas, {len(df.columns)} variables")

# ============================================================
# 1. VERIFICAR MISSING POR VARIABLE
# ============================================================

print("\n" + "="*70)
print("1. VALORES FALTANTES (MISSING) POR VARIABLE")
print("="*70)

missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Faltantes': missing,
    'Porcentaje': missing_pct
})
missing_df = missing_df[missing_df['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

if len(missing_df) == 0:
    print("✅ ¡No hay valores faltantes en ninguna variable!")
else:
    print(missing_df)

# ============================================================
# 2. VERIFICAR MISSING EN VARIABLES CLAVE (GASTO)
# ============================================================

print("\n" + "="*70)
print("2. MISSING EN VARIABLES DE GASTO")
print("="*70)

vars_gasto = ['gasto_total_hogar', 'gasto_percapita', 'log_gasto_percapita']

for var in vars_gasto:
    if var in df.columns:
        n_missing = df[var].isna().sum()
        pct_missing = n_missing / len(df) * 100
        print(f"  • {var}: {n_missing:,} missing ({pct_missing:.2f}%)")
    else:
        print(f"  ❌ {var} NO encontrada")

# ============================================================
# 3. VERIFICAR MISSING EN VARIABLES DEL MODELO
# ============================================================

print("\n" + "="*70)
print("3. MISSING EN VARIABLES DEL MODELO")
print("="*70)

vars_modelo = ['usa_billetera', 'trabajador_informal', 'edad', 'sexo', 
               'nivel_educativo', 'estrato', 'dominio', 'miembros_hogar',
               'log_gasto_percapita']

for var in vars_modelo:
    if var in df.columns:
        n_missing = df[var].isna().sum()
        pct_missing = n_missing / len(df) * 100
        print(f"  • {var}: {n_missing:,} missing ({pct_missing:.2f}%)")
    else:
        print(f"  ❌ {var} NO encontrada")

# ============================================================
# 4. ESTADÍSTICAS DE GASTO
# ============================================================

print("\n" + "="*70)
print("4. ESTADÍSTICAS DESCRIPTIVAS DE GASTO")
print("="*70)

for var in ['gasto_total_hogar', 'gasto_percapita', 'log_gasto_percapita']:
    if var in df.columns:
        print(f"\n📊 {var}:")
        print(f"  • Media: {df[var].mean():.2f}")
        print(f"  • Mediana: {df[var].median():.2f}")
        print(f"  • Mínimo: {df[var].min():.2f}")
        print(f"  • Máximo: {df[var].max():.2f}")
        print(f"  • Desv. estándar: {df[var].std():.2f}")

# ============================================================
# 5. FILAS COMPLETAS (sin missing en todo el modelo)
# ============================================================

print("\n" + "="*70)
print("5. FILAS COMPLETAS PARA EL MODELO")
print("="*70)

vars_completas = ['usa_billetera', 'trabajador_informal', 'edad', 'sexo', 
                  'nivel_educativo', 'estrato', 'dominio', 'miembros_hogar',
                  'log_gasto_percapita', 'nuevo_credito_formal']

completas = df.dropna(subset=vars_completas).shape[0]
print(f"  • Filas completas (sin missing): {completas:,} de {len(df):,} ({completas/len(df)*100:.1f}%)")
print(f"  • Filas con algún missing: {len(df) - completas:,} ({(len(df)-completas)/len(df)*100:.1f}%)")

ANÁLISIS DE MISSING - BASE_REGRESIONES_CON_GASTO.csv
✅ Base cargada: 6,358 filas, 21 variables

1. VALORES FALTANTES (MISSING) POR VARIABLE
                     Faltantes  Porcentaje
gasto_total_hogar         6346   99.811261
gasto_percapita           6346   99.811261
log_gasto_percapita       6346   99.811261

2. MISSING EN VARIABLES DE GASTO
  • gasto_total_hogar: 6,346 missing (99.81%)
  • gasto_percapita: 6,346 missing (99.81%)
  • log_gasto_percapita: 6,346 missing (99.81%)

3. MISSING EN VARIABLES DEL MODELO
  • usa_billetera: 0 missing (0.00%)
  • trabajador_informal: 0 missing (0.00%)
  • edad: 0 missing (0.00%)
  • sexo: 0 missing (0.00%)
  • nivel_educativo: 0 missing (0.00%)
  • estrato: 0 missing (0.00%)
  • dominio: 0 missing (0.00%)
  • miembros_hogar: 0 missing (0.00%)
  • log_gasto_percapita: 6,346 missing (99.81%)

4. ESTADÍSTICAS DESCRIPTIVAS DE GASTO

📊 gasto_total_hogar:
  • Media: 33723.83
  • Mediana: 33274.00
  • Mínimo: 16182.00
  • Máximo: 57303.00
  • Desv. es

In [4]:
# -*- coding: utf-8 -*-
"""
VERIFICAR DISPONIBILIDAD DE GASHOG2 EN MASTER_2024 y MASTER_2025
"""

import pandas as pd
import os

base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

print("="*70)
print("VERIFICANDO GASHOG2 EN MASTER")
print("="*70)

# ============================================================
# 1. VERIFICAR MASTER_2024
# ============================================================

df24 = pd.read_csv(os.path.join(base_resultados, "MASTER_2024.csv"), encoding="utf-8-sig", low_memory=False)

print("\n" + "="*70)
print("MASTER_2024")
print("="*70)

print(f"📊 Total de observaciones: {len(df24):,}")

if 'GASHOG2' in df24.columns:
    # Convertir a numérico
    df24['GASHOG2_num'] = pd.to_numeric(df24['GASHOG2'], errors='coerce')
    
    n_con_gasto = df24['GASHOG2_num'].notna().sum()
    n_sin_gasto = len(df24) - n_con_gasto
    
    print(f"\n📊 GASHOG2 en MASTER_2024:")
    print(f"  • Con datos (no missing): {n_con_gasto:,} ({n_con_gasto/len(df24)*100:.1f}%)")
    print(f"  • Sin datos (missing): {n_sin_gasto:,} ({n_sin_gasto/len(df24)*100:.1f}%)")
    
    # Ver si hay relación con jefatura
    if 'P203' in df24.columns:
        df24['P203_num'] = pd.to_numeric(df24['P203'], errors='coerce')
        
        # Jefes de hogar
        mask_jefe = df24['P203_num'] == 1
        jefes_total = mask_jefe.sum()
        jefes_con_gasto = (mask_jefe & df24['GASHOG2_num'].notna()).sum()
        
        print(f"\n📊 Jefes de hogar en MASTER_2024:")
        print(f"  • Total jefes: {jefes_total:,}")
        print(f"  • Jefes con GASHOG2: {jefes_con_gasto:,} ({jefes_con_gasto/jefes_total*100:.1f}%)")
else:
    print("❌ GASHOG2 NO está en MASTER_2024")

# ============================================================
# 2. VERIFICAR MASTER_2025
# ============================================================

df25 = pd.read_csv(os.path.join(base_resultados, "MASTER_2025.csv"), encoding="utf-8-sig", low_memory=False)

print("\n" + "="*70)
print("MASTER_2025")
print("="*70)

print(f"📊 Total de observaciones: {len(df25):,}")

if 'GASHOG2' in df25.columns:
    # Convertir a numérico
    df25['GASHOG2_num'] = pd.to_numeric(df25['GASHOG2'], errors='coerce')
    
    n_con_gasto = df25['GASHOG2_num'].notna().sum()
    n_sin_gasto = len(df25) - n_con_gasto
    
    print(f"\n📊 GASHOG2 en MASTER_2025:")
    print(f"  • Con datos (no missing): {n_con_gasto:,} ({n_con_gasto/len(df25)*100:.1f}%)")
    print(f"  • Sin datos (missing): {n_sin_gasto:,} ({n_sin_gasto/len(df25)*100:.1f}%)")
    
    # Ver si hay relación con jefatura
    if 'P203' in df25.columns:
        df25['P203_num'] = pd.to_numeric(df25['P203'], errors='coerce')
        
        # Jefes de hogar
        mask_jefe = df25['P203_num'] == 1
        jefes_total = mask_jefe.sum()
        jefes_con_gasto = (mask_jefe & df25['GASHOG2_num'].notna()).sum()
        
        print(f"\n📊 Jefes de hogar en MASTER_2025:")
        print(f"  • Total jefes: {jefes_total:,}")
        print(f"  • Jefes con GASHOG2: {jefes_con_gasto:,} ({jefes_con_gasto/jefes_total*100:.1f}%)")
else:
    print("❌ GASHOG2 NO está en MASTER_2025")

# ============================================================
# 3. VERIFICAR EN TU MUESTRA FINAL (EMPAREJADA)
# ============================================================

print("\n" + "="*70)
print("COMPARACIÓN CON MUESTRA FINAL")
print("="*70)

# Cargar la base con gasto que creaste
df_final = pd.read_csv(os.path.join(base_resultados, "BASE_REGRESIONES_CON_GASTO.csv"), encoding="utf-8-sig")

print(f"📊 BASE_REGRESIONES_CON_GASTO.csv:")
print(f"  • Total: {len(df_final):,} observaciones")
print(f"  • Con gasto_total_hogar: {df_final['gasto_total_hogar'].notna().sum():,} ({df_final['gasto_total_hogar'].notna().sum()/len(df_final)*100:.1f}%)")
print(f"  • Sin gasto_total_hogar: {df_final['gasto_total_hogar'].isna().sum():,} ({df_final['gasto_total_hogar'].isna().sum()/len(df_final)*100:.1f}%)")

# ============================================================
# 4. VERIFICAR EL MERGE (¿por qué se pierden?)
# ============================================================

print("\n" + "="*70)
print("VERIFICANDO EL MERGE - ¿POR QUÉ SE PIERDEN?")
print("="*70)

# Crear llave de hogar en MASTER_2024
def crear_llave_hogar(df):
    df = df.copy()
    for c in ["CONGLOME", "VIVIENDA", "HOGAR"]:
        df[f"_{c}"] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
    df["llave_hogar"] = df[["_CONGLOME", "_VIVIENDA", "_HOGAR"]].agg("-".join, axis=1)
    return df

df24_hogar = crear_llave_hogar(df24)

# Contar hogares con gasto en MASTER_2024
if 'GASHOG2' in df24_hogar.columns:
    df24_hogar['GASHOG2_num'] = pd.to_numeric(df24_hogar['GASHOG2'], errors='coerce')
    
    # Hogares únicos con gasto
    hogares_con_gasto = df24_hogar[df24_hogar['GASHOG2_num'].notna()]['llave_hogar'].nunique()
    hogares_totales = df24_hogar['llave_hogar'].nunique()
    
    print(f"\n📊 Hogares en MASTER_2024:")
    print(f"  • Total hogares: {hogares_totales:,}")
    print(f"  • Hogares con GASHOG2: {hogares_con_gasto:,} ({hogares_con_gasto/hogares_totales*100:.1f}%)")

VERIFICANDO GASHOG2 EN MASTER

MASTER_2024
📊 Total de observaciones: 117,721

📊 GASHOG2 en MASTER_2024:
  • Con datos (no missing): 295 (0.3%)
  • Sin datos (missing): 117,426 (99.7%)

📊 Jefes de hogar en MASTER_2024:
  • Total jefes: 33,691
  • Jefes con GASHOG2: 83 (0.2%)

MASTER_2025
📊 Total de observaciones: 115,145

📊 GASHOG2 en MASTER_2025:
  • Con datos (no missing): 327 (0.3%)
  • Sin datos (missing): 114,818 (99.7%)

📊 Jefes de hogar en MASTER_2025:
  • Total jefes: 33,702
  • Jefes con GASHOG2: 85 (0.3%)

COMPARACIÓN CON MUESTRA FINAL
📊 BASE_REGRESIONES_CON_GASTO.csv:
  • Total: 6,358 observaciones
  • Con gasto_total_hogar: 12 (0.2%)
  • Sin gasto_total_hogar: 6,346 (99.8%)

VERIFICANDO EL MERGE - ¿POR QUÉ SE PIERDEN?

📊 Hogares en MASTER_2024:
  • Total hogares: 33,691
  • Hogares con GASHOG2: 83 (0.2%)


In [5]:
# -*- coding: utf-8 -*-
"""
EXPLORAR TODAS LAS VARIABLES DE GASTO E INGRESO EN ARCHIVOS ORIGINALES
"""

import pandas as pd
import os
import numpy as np

# ============================================================
# CONFIGURACIÓN
# ============================================================
datos_originales = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO"
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def encontrar_archivo(carpeta, inicio_nombre):
    for f in os.listdir(carpeta):
        if f.upper().replace(".CSV", "").startswith(inicio_nombre.upper()):
            return os.path.join(carpeta, f)
    return None

def leer_columnas(ruta):
    try:
        for sep in [",", ";"]:
            for enc in ["utf-8-sig", "latin1"]:
                try:
                    df_sample = pd.read_csv(ruta, sep=sep, encoding=enc, nrows=0)
                    return df_sample.columns.tolist()
                except:
                    pass
    except:
        pass
    return None

def leer_muestra(ruta, cols, n=1000):
    try:
        for sep in [",", ";"]:
            for enc in ["utf-8-sig", "latin1"]:
                try:
                    df = pd.read_csv(ruta, sep=sep, encoding=enc, usecols=cols, nrows=n, low_memory=False)
                    return df
                except:
                    pass
    except:
        pass
    return None

# ============================================================
# 1. EXPLORAR MÓDULO SUMARIA (GASTO) - 2024
# ============================================================

print("="*70)
print("1. EXPLORANDO MÓDULO SUMARIA 2024")
print("="*70)

carpeta_sumaria_2024 = os.path.join(datos_originales, "966-Modulo34")
archivo_sumaria = encontrar_archivo(carpeta_sumaria_2024, "Sumaria")

if archivo_sumaria:
    print(f"✅ Archivo encontrado: {archivo_sumaria}")
    
    columnas = leer_columnas(archivo_sumaria)
    if columnas:
        print(f"✅ {len(columnas)} columnas encontradas")
        
        # Buscar TODAS las variables de gasto
        vars_gasto = [c for c in columnas if 'GAS' in c or 'GAST' in c or 'GRU' in c or 'GASHOG' in c]
        
        print(f"\n📊 Variables de GASTO encontradas ({len(vars_gasto)}):")
        for v in sorted(vars_gasto):
            print(f"  • {v}")
        
        # Verificar cobertura de cada variable de gasto
        print(f"\n📊 COBERTURA DE VARIABLES DE GASTO (muestra de 1000):")
        
        # Leer muestra de datos
        df_muestra = leer_muestra(archivo_sumaria, vars_gasto, n=1000)
        
        if df_muestra is not None:
            for v in vars_gasto:
                if v in df_muestra.columns:
                    # Convertir a numérico
                    df_muestra[f"{v}_num"] = pd.to_numeric(df_muestra[v], errors='coerce')
                    n_con = df_muestra[f"{v}_num"].notna().sum()
                    pct = n_con / len(df_muestra) * 100
                    print(f"  • {v}: {n_con}/{len(df_muestra)} ({pct:.1f}%)")
        else:
            print("  ⚠️ No se pudo leer muestra")
else:
    print("❌ No se encontró archivo Sumaria 2024")

# ============================================================
# 2. EXPLORAR MÓDULO 500 (INGRESOS) - 2024
# ============================================================

print("\n" + "="*70)
print("2. EXPLORANDO MÓDULO 500 2024 (INGRESOS)")
print("="*70)

carpeta_500_2024 = os.path.join(datos_originales, "966-Modulo05")
archivo_500 = encontrar_archivo(carpeta_500_2024, "Enaho01a-2024-500")

if archivo_500:
    print(f"✅ Archivo encontrado: {archivo_500}")
    
    columnas = leer_columnas(archivo_500)
    if columnas:
        print(f"✅ {len(columnas)} columnas encontradas")
        
        # Buscar variables de ingreso (P558)
        vars_ingreso = [c for c in columnas if 'P558' in c or 'ING' in c or 'ingreso' in c.lower()]
        
        # Filtrar solo las que parecen ser de ingreso total
        vars_ingreso_total = [c for c in vars_ingreso if 'T' in c or 'total' in c.lower() or 'ING' in c]
        
        print(f"\n📊 Variables de INGRESO encontradas ({len(vars_ingreso_total)}):")
        for v in sorted(vars_ingreso_total):
            print(f"  • {v}")
        
        # Verificar cobertura de variables de ingreso clave
        print(f"\n📊 COBERTURA DE VARIABLES DE INGRESO (muestra de 1000):")
        
        vars_clave = ['P558T', 'P558T1', 'P558A1', 'P558A2', 'P558A3']
        vars_disponibles = [v for v in vars_clave if v in columnas]
        
        if vars_disponibles:
            df_muestra = leer_muestra(archivo_500, vars_disponibles, n=1000)
            
            if df_muestra is not None:
                for v in vars_disponibles:
                    if v in df_muestra.columns:
                        df_muestra[f"{v}_num"] = pd.to_numeric(df_muestra[v], errors='coerce')
                        n_con = df_muestra[f"{v}_num"].notna().sum()
                        pct = n_con / len(df_muestra) * 100
                        print(f"  • {v}: {n_con}/{len(df_muestra)} ({pct:.1f}%)")
        else:
            print("  ⚠️ No se encontraron variables de ingreso clave")
else:
    print("❌ No se encontró archivo módulo 500 2024")

# ============================================================
# 3. RESULTADO: ¿QUÉ VARIABLE USAR?
# ============================================================

print("\n" + "="*70)
print("3. RECOMENDACIÓN")
print("="*70)

print("""
📌 Si GASHOG2 tiene poca cobertura (0.2%):
   → Buscar variables de INGRESO en módulo 500 (P558T, P558T1)
   → Estas suelen tener mayor cobertura

📌 Si ninguna variable de ingreso tiene buena cobertura:
   → Usar proxies de nivel socioeconómico:
      - nivel_educativo (ya disponible)
      - estrato (ya disponible)  
      - dominio (ya disponible)
      - miembros_hogar (ya disponible)

📌 Recomendación final:
   → Si P558T o P558T1 tienen >50% de cobertura en tu muestra,
     usar log(ingreso_percapita) en lugar de log(gasto_percapita)
""")

1. EXPLORANDO MÓDULO SUMARIA 2024
✅ Archivo encontrado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO\966-Modulo34\Sumaria-2024-12g.csv
✅ 1 columnas encontradas

📊 Variables de GASTO encontradas (1):
  • AÑO;MES;CONGLOME;VIVIENDA;HOGAR;UBIGEO;DOMINIO;ESTRATO;MIEPERHO;TOTMIEHO;PERCEPHO;IA01HD;IA02HD;IG03HD1;IG03HD2;IG03HD3;IG03HD4;SG23;SIG24;SG25;SIG26;SG27;SIG28;GA03HD;GA04HD;SG42;SG421;SG422;SG42D;SG42D1;SG42D2;INGBRUHD;INGNETHD;PAGESPHD;INGINDHD;INGAUTHD;INSEDTHD;INSEDLHD;PAESECHD;INGSEIHD;ISECAUHD;INGEXTHD;INGTRAHD;INGTEXHD;INGRENHD;INGOEXHD;G05HD;IG06HD;G05HD1;IG06HD1;G05HD2;IG06HD2;G05HD3;IG06HD3;G05HD4;IG06HD4;G05HD5;IG06HD5;G05HD6;IG06HD6;G07HD1;G07HD2;IG08HD1;IG08HD2;INGTPRHD;INGTPUHD;INGTPU01;INGTPU02;INGTPU03;INGTPU04;INGTPU05;INGTPU10;INGTPU11;INGTPU12;INGTPU13;INGTPU14;INGTPU15;INGTPU16;GRU11HD;GRU12HD;GRU13HD;GRU14HD;GRU15HD;GRU16HD;GRU10HD;GRU111HD;GRU112HD;GRU113HD;GRU114HD;GRU115HD;GRU116HD;GRU110HD;GRU111HD2;GRU112HD2;GRU113HD2;GRU114HD2;G